***Importation des différentes libraires***

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

***Lire les fichiers CSV***

In [ ]:
df_agents = pd.read_csv("./agents.csv")
df_customers = pd.read_csv("./customers.csv")
df_fx = pd.read_csv("./fx_rates.csv")
df_operators = pd.read_csv("./operators.csv")
df_transactions = pd.read_csv("./transactions.csv")

***Vérifier les fichiers et leur structure***

In [ ]:
datasets = {
    "transactions": df_transactions,
    "agents": df_agents,
    "customers": df_customers,
    "operators": df_operators,
    "fx_rates": df_fx
}

for nom, df in datasets.items():
    print(f"\n--- {nom.upper()} ---")
    print("Nombre de lignes :", df.shape[0])
    print("Nombre de colonnes :", df.shape[1])
    print("Colonnes :", df.columns.tolist())
    print("Mémoire utilisée (octets) :", df.memory_usage(deep=True).sum())


In [ ]:
for nom, df in datasets.items():
    print(f"\n--- Valeurs manquantes : {nom} ---")

    resultat = pd.DataFrame({
        "valeurs_manquantes": df.isna().sum(),
        "pourcentage": (df.isna().mean() * 100).round(2),
        "type": df.dtypes.astype(str)
    })

    print(resultat)


***Mesurer les doublons techniques***

In [ ]:
controles = [
    ("transactions", df_transactions, "transaction_id"),
    ("customers", df_customers, "customer_id"),
    ("agents", df_agents, "agent_id")
]

for nom, df, identifiant in controles:
    print(f"\n--- Doublons : {nom} ---")

    if identifiant not in df.columns:
        print(f"Colonne absente : {identifiant}")
        continue

    masque = df[identifiant].notna() & df[identifiant].duplicated(keep=False)

    print("Nombre de lignes concernées :", int(masque.sum()))
    print("Nombre d'identifiants répétés :",
          df.loc[masque, identifiant].nunique())
    print("Taux de lignes concernées (%) :",
          round(masque.sum() / len(df) * 100, 2) if len(df) else 0)

    display(df.loc[masque].sort_values(identifiant).head(20))


***Analyser les montants des transactions***

In [ ]:
print(df_transactions.columns.tolist())
print(df_operators.columns.tolist())

col_montant = "amount"

if col_montant in df_transactions.columns:
    df_transactions[col_montant] = pd.to_numeric(
        df_transactions[col_montant],
        errors="coerce"
    )

    print(df_transactions[col_montant].describe())

    print("Montants manquants :",
          df_transactions[col_montant].isna().sum())

    print("Montants négatifs :",
          (df_transactions[col_montant] < 0).sum())

    print("Montants nuls :",
          (df_transactions[col_montant] == 0).sum())

    df_transactions[col_montant].hist(bins=50)
    plt.title("Distribution des montants des transactions")
    plt.xlabel("Montant")
    plt.ylabel("Nombre de transactions")
    plt.show()


***Vérifier les formats des MSISDN des émetteurs***

In [ ]:
col_msisdn = "sender_msisdn"

if col_msisdn in df_transactions.columns:
    numeros = df_transactions[col_msisdn].astype("string").str.strip()

    print("Valeurs manquantes :", numeros.isna().sum())
    print("Valeurs avec espaces :",
          numeros.str.contains(r"\s", na=False).sum())
    print("Valeurs commençant par + :",
          numeros.str.startswith("+", na=False).sum())
    print("Valeurs commençant par 00 :",
          numeros.str.startswith("00", na=False).sum())

    print("\nExemples de formats :")
    display(numeros.value_counts().head(20))

***Vérifier les formats des MSISDN des récepteurs***

In [ ]:
col_msisdn = "receiver_msisdn"

if col_msisdn in df_transactions.columns:
    numeros = df_transactions[col_msisdn].astype("string").str.strip()

    print("Valeurs manquantes :", numeros.isna().sum())
    print("Valeurs avec espaces :",
          numeros.str.contains(r"\s", na=False).sum())
    print("Valeurs commençant par + :",
          numeros.str.startswith("+", na=False).sum())
    print("Valeurs commençant par 00 :",
          numeros.str.startswith("00", na=False).sum())

    print("\nExemples de formats :")
    display(numeros.value_counts().head(20))

***Vérifier l'intégrité référentielle***

In [ ]:
# 3) Normalisation syntaxique des MSISDN d'émetteurs et de récepteurs
def normaliser_msisdn(serie):
    s = serie.astype("string").str.strip()
    s = s.str.replace(r"\s+", "", regex=True)
    s = s.str.replace(r"^00", "+", regex=True)
    return s

def masque_format_e164(serie_normalisee):
    # E.164 : signe + suivi de 2 à 15 chiffres au total, premier chiffre non nul.
    return serie_normalisee.str.fullmatch(r"\+[1-9]\d{1,14}", na=False)

for colonne in ["sender_msisdn", "receiver_msisdn"]:
    if colonne not in df_transactions.columns:
        print(f"{colonne} : colonne absente")
        continue

    original = df_transactions[colonne].astype("string")
    normalise = normaliser_msisdn(original)
    non_vide = original.notna() & original.str.strip().ne("")
    espaces = original.str.contains(r"\s", na=False)
    prefixe_00 = original.str.strip().str.startswith("00", na=False)
    prefixe_plus = original.str.strip().str.startswith("+", na=False)
    format_invalide_apres_normalisation = non_vide & ~masque_format_e164(normalise)

    print(f"\n--- {colonne} ---")
    print("Valeurs manquantes/vides :", int((~non_vide).sum()))
    print("Valeurs contenant des espaces :", int(espaces.sum()))
    print("Valeurs commençant par 00 :", int(prefixe_00.sum()))
    print("Valeurs commençant par + :", int(prefixe_plus.sum()))
    print("Valeurs invalides selon le contrôle syntaxique E.164 après normalisation :",
          int(format_invalide_apres_normalisation.sum()))
    print("Taux invalide parmi les valeurs non vides :",
          round(100 * format_invalide_apres_normalisation.sum() / non_vide.sum(), 2)
          if non_vide.sum() else 0, "%")

    display(pd.DataFrame({
        "original": original[format_invalide_apres_normalisation],
        "normalise": normalise[format_invalide_apres_normalisation]
    }).head(10))

In [ ]:
# 5) Tableau récapitulatif des contrôles disponibles
n_tx = len(df_transactions)

def taux_pct(n, denom):
    return round(100 * n / denom, 2) if denom else np.nan

synthese = []

# Les copies excédentaires et les lignes concernées sont deux mesures distinctes.
if "transaction_id" in df_transactions.columns:
    ids_tx = df_transactions["transaction_id"]
    copies = int((ids_tx.notna() & ids_tx.duplicated()).sum())
    lignes_doublons = int(
        (ids_tx.notna() & ids_tx.duplicated(keep=False)).sum()
    )
    synthese.append({
        "controle": "transaction_id répété",
        "nombre": copies,
        "taux_pct": taux_pct(copies, n_tx),
        "attendu": "environ 5 % (définition à confirmer)",
    })
    synthese.append({
        "controle": "lignes dans groupes de transaction_id répétés",
        "nombre": lignes_doublons,
        "taux_pct": taux_pct(lignes_doublons, n_tx),
        "attendu": "métrique différente du nombre de copies",
    })

# Montants négatifs
if "amount" in df_transactions.columns:
    montants = pd.to_numeric(df_transactions["amount"], errors="coerce")
    n_neg = int((montants < 0).sum())
    synthese.append({
        "controle": "montants négatifs",
        "nombre": n_neg,
        "taux_pct": taux_pct(n_neg, n_tx),
        "attendu": "inclus dans environ 3 % de montants invalides",
    })

# MSISDN invalides après normalisation
for colonne in ["sender_msisdn", "receiver_msisdn"]:
    if colonne in df_transactions.columns:
        original = df_transactions[colonne].astype("string")
        normalise = normaliser_msisdn(original)
        non_vide = original.notna() & original.str.strip().ne("")
        invalides = non_vide & ~masque_format_e164(normalise)
        n_invalides = int(invalides.sum())
        synthese.append({
            "controle": f"{colonne} invalide après normalisation E.164",
            "nombre": n_invalides,
            "taux_pct": taux_pct(n_invalides, int(non_vide.sum())),
            "attendu": "à comparer aux ~2,5 % après validation",
        })

# Agent inconnu
if "agent_id" in df_transactions.columns and "agent_id" in df_agents.columns:
    agents_connus = set(df_agents["agent_id"].dropna())
    agent_renseigne = df_transactions["agent_id"].notna()
    masque = agent_renseigne & ~df_transactions["agent_id"].isin(
        agents_connus
    )
    n_bad_agent = int(masque.sum())
    synthese.append({
        "controle": "agent_id inexistant",
        "nombre": n_bad_agent,
        "taux_pct": taux_pct(n_bad_agent, n_tx),
        "attendu": "environ 1 %",
    })

# Dates de naissance sentinelles
if "birth_date" in df_customers.columns:
    sentinelles = {
        "1900-01-01", "9999-12-31", "0000-00-00", "1850-05-12"
    }
    dates = df_customers["birth_date"].astype("string").str.strip()
    n_dates = int(dates.isin(sentinelles).sum())
    synthese.append({
        "controle": "dates de naissance sentinelles",
        "nombre": n_dates,
        "taux_pct": taux_pct(n_dates, len(df_customers)),
        "attendu": "environ 0,5 %",
    })

# Coordonnées manquantes et hors limites, séparément
if {"latitude", "longitude"}.issubset(df_agents.columns):
    lat = pd.to_numeric(df_agents["latitude"], errors="coerce")
    lon = pd.to_numeric(df_agents["longitude"], errors="coerce")
    missing_geo = lat.isna() | lon.isna()
    out_of_range = (
        lat.notna() & lon.notna()
        & (~lat.between(-90, 90) | ~lon.between(-180, 180))
    )
    synthese.append({
        "controle": "agents sans coordonnées complètes",
        "nombre": int(missing_geo.sum()),
        "taux_pct": taux_pct(int(missing_geo.sum()), len(df_agents)),
        "attendu": "quelques cas",
    })
    synthese.append({
        "controle": "agents avec coordonnées présentes hors limites",
        "nombre": int(out_of_range.sum()),
        "taux_pct": taux_pct(int(out_of_range.sum()), len(df_agents)),
        "attendu": "à vérifier séparément",
    })

# Copies excédentaires dans les référentiels
for nom, df, col, attendu in [
    ("clients", df_customers, "customer_id", "50 doublons techniques"),
    ("agents", df_agents, "agent_id", "5 doublons techniques"),
]:
    if col in df.columns:
        serie = df[col]
        n = int((serie.notna() & serie.duplicated()).sum())
        synthese.append({
            "controle": f"copies excédentaires {nom}",
            "nombre": n,
            "taux_pct": taux_pct(n, len(df)),
            "attendu": attendu,
        })

synthese_eda = pd.DataFrame(synthese)
display(synthese_eda)

***Contrôler les dates de naissance***

In [ ]:
col_date = "birth_date"

if col_date in df_customers.columns:
    dates_suspectes = [
        "1900-01-01",
        "9999-12-31",
        "0000-00-00",
        "1850-05-12"
    ]

    dates = df_customers[col_date].astype("string").str.strip()

    masque_suspect = dates.isin(dates_suspectes)

    dates_converties = pd.to_datetime(
        dates,
        errors="coerce"
    )

    print("Dates sentinelles :", masque_suspect.sum())
    print("Dates non convertibles :",
          dates_converties.isna().sum())

    display(df_customers.loc[masque_suspect])

***Examiner les coordonnées géographiques des agents***

In [ ]:
colonnes_geo = ["latitude", "longitude"]

if all(c in df_agents.columns for c in colonnes_geo):
    agents_geo = df_agents.copy()

    for col in colonnes_geo:
        agents_geo[col] = pd.to_numeric(agents_geo[col], errors="coerce")

    # Cas 1 : au moins une coordonnée manquante ou non numérique
    masque_geo_absente = agents_geo[colonnes_geo].isna().any(axis=1)

    # Cas 2 : coordonnées présentes, mais hors des limites géographiques
    masque_geo_hors_limites = (
        agents_geo["latitude"].notna()
        & agents_geo["longitude"].notna()
        & (
            ~agents_geo["latitude"].between(-90, 90)
            | ~agents_geo["longitude"].between(-180, 180)
        )
    )

    print("Agents sans coordonnées complètes :", int(masque_geo_absente.sum()))
    print("Agents avec coordonnées présentes mais hors limites :",
          int(masque_geo_hors_limites.sum()))

    display(agents_geo.loc[masque_geo_absente].head(20))
    display(agents_geo.loc[masque_geo_hors_limites].head(20))
else:
    print("Colonnes géographiques manquantes :",
          [c for c in colonnes_geo if c not in df_agents.columns])

***Détecter les transactions miroir***

In [ ]:
col_source = "sender_msisdn"
col_destination = "receiver_msisdn"
col_montant = "amount"
col_date_transaction = "initiated_at"

colonnes_requises = [
    col_source,
    col_destination,
    col_montant,
    col_date_transaction
]

if all(c in df_transactions.columns for c in colonnes_requises):

    tx = df_transactions.copy()

    tx[col_date_transaction] = pd.to_datetime(
        tx[col_date_transaction],
        errors="coerce",
        utc=True
    )

    tx[col_montant] = pd.to_numeric(
        tx[col_montant],
        errors="coerce"
    )

    tx = tx.dropna(subset=colonnes_requises)

    inverses = tx[
        [
            "transaction_id",
            col_source,
            col_destination,
            col_montant,
            col_date_transaction
        ]
    ].copy()

    inverses = inverses.rename(columns={
        "transaction_id": "transaction_id_B",
        col_source: "receiver_B",
        col_destination: "sender_B",
        col_montant: "amount_B",
        col_date_transaction: "timestamp_B"
    })

    paires = tx.merge(
        inverses,
        left_on=[
            col_source,
            col_destination,
            col_montant
        ],
        right_on=[
            "sender_B",
            "receiver_B",
            "amount_B"
        ]
    )

    ecart = (
        paires[col_date_transaction] - paires["timestamp_B"]
    ).abs()

    paires = paires[
        (ecart < pd.Timedelta(seconds=60))
        & (
            paires["transaction_id"]
            != paires["transaction_id_B"]
        )
    ].copy()

    paires["paire_id"] = paires.apply(
        lambda ligne: tuple(sorted([
            ligne["transaction_id"],
            ligne["transaction_id_B"]
        ])),
        axis=1
    )

    paires = paires.drop_duplicates(subset="paire_id")

    print("Nombre de paires miroir candidates :", len(paires))

    display(paires[[
        "transaction_id",
        "transaction_id_B",
        col_source,
        col_destination,
        col_montant,
        col_date_transaction,
        "timestamp_B"
    ]].head(20))

else:
    print(
        "Colonnes manquantes :",
        [c for c in colonnes_requises
         if c not in df_transactions.columns]
    )


***Détecter une vélocité anormale***

In [ ]:
# Le fichier transactions.csv ne contient pas customer_id.
# On utilise sender_msisdn comme proxy du client émetteur.
col_client = "sender_msisdn"
col_temps = "initiated_at"
seuil = 25
duree = pd.Timedelta(minutes=10)

colonnes_requises = [col_client, col_temps]

if all(c in df_transactions.columns for c in colonnes_requises):
    tx_velocite = df_transactions.copy()
    tx_velocite[col_temps] = pd.to_datetime(
        tx_velocite[col_temps], errors="coerce", utc=True
    )
    tx_velocite = tx_velocite.dropna(
        subset=[col_client, col_temps]
    ).sort_values([col_client, col_temps])

    resultats = []
    for client, groupe in tx_velocite.groupby(col_client, sort=False):
        temps = groupe[col_temps].reset_index(drop=True)
        debut = 0
        for fin in range(len(temps)):
            while temps.iloc[fin] - temps.iloc[debut] >= duree:
                debut += 1
            nombre = fin - debut + 1
            if nombre >= seuil:
                resultats.append({
                    "client_msisdn": client,
                    "debut_fenetre": temps.iloc[debut],
                    "fin_fenetre": temps.iloc[fin],
                    "nombre_transactions": nombre
                })

    suspects_velocite = pd.DataFrame(resultats)
    print("Fenêtres suspectes détectées :", len(suspects_velocite))
    if not suspects_velocite.empty:
        # Plusieurs fenêtres peuvent correspondre à une même séquence.
        display(suspects_velocite.head(20))
    else:
        print("Aucune fenêtre glissante ne dépasse le seuil.")
else:
    print("Colonnes manquantes :",
          [c for c in colonnes_requises
           if c not in df_transactions.columns])

## Compléments EDA — contrôles métier, synthèse et interprétation

Cette partie complète les contrôles précédents. Elle mesure séparément les copies excédentaires et les lignes concernées par les doublons, normalise les formats de téléphone pour l'analyse, vérifie les références et produit un tableau de synthèse.

**Important :** les résultats réellement calculés dépendront des CSV présents dans le répertoire du notebook. Le plafond réglementaire par transaction n'est pas explicitement fourni dans `operators.csv` : les limites disponibles sont quotidiennes et mensuelles. Elles ne doivent donc pas être présentées comme un plafond unitaire par transaction.

In [ ]:
# 1) Distinguer copies excédentaires et lignes concernées par les doublons
def resume_doublons(df, colonne, nom):
    if colonne not in df.columns:
        return {
            "controle": nom,
            "mesure": "identifiant absent",
            "nombre": np.nan,
            "taux_pct": np.nan,
        }

    serie = df[colonne]
    masque_repetitions = (
        serie.notna() & serie.duplicated(keep="first")
    )
    masque_groupes = serie.notna() & serie.duplicated(keep=False)
    n_lignes = len(df)

    return {
        "controle": nom,
        "lignes_totales": n_lignes,
        "copies_excedentaires": int(masque_repetitions.sum()),
        "identifiants_repetes_distincts": int(
            serie[masque_groupes].nunique()
        ),
        "lignes_dans_groupes_repetes": int(masque_groupes.sum()),
        "taux_copies_excedentaires_pct": (
            round(100 * masque_repetitions.sum() / n_lignes, 3)
            if n_lignes else 0
        ),
        "taux_lignes_concernees_pct": (
            round(100 * masque_groupes.sum() / n_lignes, 3)
            if n_lignes else 0
        ),
    }

resume_doublons_df = pd.DataFrame([
    resume_doublons(
        df_transactions, "transaction_id", "Doublons transaction_id"
    ),
    resume_doublons(df_customers, "customer_id", "Doublons customer_id"),
    resume_doublons(df_agents, "agent_id", "Doublons agent_id"),
])

display(resume_doublons_df)

### Contrôle des montants et des limites opérateur

Le fichier des opérateurs expose des plafonds journaliers et mensuels, mais pas un plafond individuel par transaction. La cellule ci-dessous calcule les montants négatifs et effectue une comparaison indicative avec la limite journalière associée à l'opérateur. Cette comparaison est un **signal à investiguer**, pas une preuve de dépassement du plafond réglementaire d'une transaction. Vérifie aussi l'unité et la devise des plafonds dans la documentation métier.

In [ ]:
# 2) Montants négatifs, nuls et comparaison indicative avec la limite journalière
tx_montants = df_transactions.copy()
tx_montants["amount"] = pd.to_numeric(
    tx_montants["amount"], errors="coerce"
)

masque_negatif = tx_montants["amount"] < 0
masque_nul = tx_montants["amount"] == 0
masque_montant_manquant = tx_montants["amount"].isna()

print(
    "Montants négatifs :", int(masque_negatif.sum()),
    f"({masque_negatif.mean() * 100:.2f} % des transactions)"
)
print("Montants nuls :", int(masque_nul.sum()))
print("Montants manquants/non numériques :", int(masque_montant_manquant.sum()))

# La limite fournie est journalière, pas transactionnelle.
if {"operator_code", "transaction_limit_daily"}.issubset(
    df_operators.columns
):
    limites = df_operators[
        ["operator_code", "transaction_limit_daily"]
    ].copy()
    limites["transaction_limit_daily"] = pd.to_numeric(
        limites["transaction_limit_daily"], errors="coerce"
    )
    tx_limites = tx_montants.merge(
        limites, on="operator_code", how="left"
    )
    masque_limite_absente = tx_limites["transaction_limit_daily"].isna()
    masque_sup_limite_journaliere = (
        tx_limites["amount"].notna()
        & tx_limites["transaction_limit_daily"].notna()
        & (
            tx_limites["amount"]
            > tx_limites["transaction_limit_daily"]
        )
    )
    print(
        "Transactions sans limite opérateur correspondante :",
        int(masque_limite_absente.sum()),
    )
    print(
        "Montants supérieurs à la limite journalière (signal indicatif) :",
        int(masque_sup_limite_journaliere.sum()),
    )
    display(tx_limites.loc[masque_sup_limite_journaliere, [
        "transaction_id", "operator_code", "currency", "amount",
        "transaction_limit_daily",
    ]].head(20))
else:
    print(
        "Comparaison impossible : operator_code ou "
        "transaction_limit_daily absent."
    )

plt.figure(figsize=(9, 4))
montants_valides = tx_montants.loc[
    tx_montants["amount"].ge(0), "amount"
]
montants_visibles = montants_valides.clip(
    upper=montants_valides.quantile(0.99)
)
montants_visibles.hist(bins=40)
plt.title(
    "Distribution des montants non négatifs "
    "(plafonnée au 99e percentile)"
)
plt.xlabel("Montant")
plt.ylabel("Nombre de transactions")
plt.tight_layout()
plt.show()

### Normalisation exploratoire des MSISDN

Les numéros de téléphone ne doivent pas être additionnés par catégories de formats : un numéro peut contenir des espaces et commencer par `00` en même temps. La cellule crée une forme canonique exploratoire en supprimant les espaces et en convertissant le préfixe `00` en `+`. Elle identifie ensuite les formats qui ne ressemblent pas à E.164. Cette validation syntaxique ne garantit pas que le numéro existe ou soit attribué.

### Intégrité référentielle et doublons des référentiels

Le contrôle `agent_id` doit distinguer les identifiants manquants des identifiants non nuls absents du référentiel. Les contrôles suivants vérifient aussi les identifiants clients répétés et les agents répétés, sans supprimer automatiquement les lignes.

In [ ]:
# 4) Intégrité référentielle agent et client
if "agent_id" in df_transactions.columns and "agent_id" in df_agents.columns:
    agents_connus = set(df_agents["agent_id"].dropna())
    agent_renseigne = df_transactions["agent_id"].notna()
    agent_existe = df_transactions["agent_id"].isin(agents_connus)
    masque_agent_absent = agent_renseigne & ~agent_existe
    masque_agent_id_manquant = df_transactions["agent_id"].isna()

    print(
        "Transactions avec agent_id renseigné mais inexistant :",
        int(masque_agent_absent.sum()),
    )
    print(
        "Taux sur toutes les transactions :",
        round(100 * masque_agent_absent.mean(), 2), "%",
    )
    print(
        "Transactions sans agent_id :",
        int(masque_agent_id_manquant.sum()),
        "(peut être normal selon le type de transaction)",
    )
    display(df_transactions.loc[masque_agent_absent].head(20))

# Identifiants répétés à examiner avant tout dédoublonnage
for nom, df, col in [
    ("customers", df_customers, "customer_id"),
    ("agents", df_agents, "agent_id"),
]:
    if col in df.columns:
        repetitions = df.loc[
            df[col].notna() & df[col].duplicated(keep=False)
        ].sort_values(col)
        print(
            f"\n{nom} : {repetitions[col].nunique()} identifiants "
            f"répétés, {len(repetitions)} lignes concernées."
        )
        display(repetitions.head(20))

### Synthèse automatique des contrôles

Le tableau ci-dessous rassemble les résultats recalculés à partir des données chargées. Les pourcentages annoncés dans le cahier des charges sont des valeurs attendues approximatives : un écart doit être expliqué, pas forcé. Les catégories peuvent se chevaucher ; elles ne doivent donc pas être additionnées pour calculer un taux global d'anomalies.

In [ ]:
# 5) Tableau récapitulatif des contrôles disponibles
n_tx = len(df_transactions)

def taux_pct(n, denom):
    return round(100 * n / denom, 2) if denom else np.nan

synthese = []

# Doublons transaction_id : les deux mesures sont présentées pour lever toute ambiguïté.
if "transaction_id" in df_transactions.columns:
    ids_tx = df_transactions["transaction_id"]
    copies = int((ids_tx.notna() & ids_tx.duplicated()).sum())
    lignes_doublons = int((ids_tx.notna() & ids_tx.duplicated(keep=False)).sum())
    synthese.append({
        "controle": "transaction_id répété",
        "nombre": copies,
        "taux_pct": taux_pct(copies, n_tx),
        "attendu": "environ 5 % (définition à confirmer)"
    })
    synthese.append({
        "controle": "lignes appartenant à un groupe de transaction_id répétés",
        "nombre": lignes_doublons,
        "taux_pct": taux_pct(lignes_doublons, n_tx),
        "attendu": "métrique différente du nombre de copies"
    })

# Montants négatifs
if "amount" in df_transactions.columns:
    montants = pd.to_numeric(df_transactions["amount"], errors="coerce")
    n_neg = int((montants < 0).sum())
    synthese.append({
        "controle": "montants négatifs",
        "nombre": n_neg,
        "taux_pct": taux_pct(n_neg, n_tx),
        "attendu": "inclus dans environ 3 % de montants invalides"
    })

# MSISDN syntaxiquement invalides après normalisation
for colonne in ["sender_msisdn", "receiver_msisdn"]:
    if colonne in df_transactions.columns:
        original = df_transactions[colonne].astype("string")
        normalise = normaliser_msisdn(original)
        non_vide = original.notna() & original.str.strip().ne("")
        invalides = non_vide & ~masque_format_e164(normalise)
        n_invalides = int(invalides.sum())
        synthese.append({
            "controle": f"{colonne} invalide après normalisation E.164",
            "nombre": n_invalides,
            "taux_pct": taux_pct(n_invalides, int(non_vide.sum())),
            "attendu": "à comparer aux ~2,5 % après validation de la règle"
        })

# Agent inconnu
if "agent_id" in df_transactions.columns and "agent_id" in df_agents.columns:
    agents_connus = set(df_agents["agent_id"].dropna())
    agent_renseigne = df_transactions["agent_id"].notna()
    masque = agent_renseigne & ~df_transactions["agent_id"].isin(
        agents_connus
    )
    n_bad_agent = int(masque.sum())
    synthese.append({
        "controle": "agent_id inexistant",
        "nombre": n_bad_agent,
        "taux_pct": taux_pct(n_bad_agent, n_tx),
        "attendu": "environ 1 %"
    })

# Dates de naissance sentinelles
if "birth_date" in df_customers.columns:
    sentinelles = {"1900-01-01", "9999-12-31", "0000-00-00", "1850-05-12"}
    dates = df_customers["birth_date"].astype("string").str.strip()
    n_dates = int(dates.isin(sentinelles).sum())
    synthese.append({
        "controle": "dates de naissance sentinelles",
        "nombre": n_dates,
        "taux_pct": taux_pct(n_dates, len(df_customers)),
        "attendu": "environ 0,5 %"
    })

# Coordonnées manquantes et limites invalides séparées
if {"latitude", "longitude"}.issubset(df_agents.columns):
    lat = pd.to_numeric(df_agents["latitude"], errors="coerce")
    lon = pd.to_numeric(df_agents["longitude"], errors="coerce")
    missing_geo = lat.isna() | lon.isna()
    out_of_range = lat.notna() & lon.notna() & (
        ~lat.between(-90, 90) | ~lon.between(-180, 180)
    )
    synthese.append({
        "controle": "agents sans coordonnées complètes",
        "nombre": int(missing_geo.sum()),
        "taux_pct": taux_pct(int(missing_geo.sum()), len(df_agents)),
        "attendu": "quelques cas"
    })
    synthese.append({
        "controle": "agents avec coordonnées présentes hors limites",
        "nombre": int(out_of_range.sum()),
        "taux_pct": taux_pct(int(out_of_range.sum()), len(df_agents)),
        "attendu": "à vérifier séparément"
    })

# Doublons clients et agents : copies excédentaires et non groupes
for nom, df, col, attendu in [
    ("clients", df_customers, "customer_id", "50 doublons techniques"),
    ("agents", df_agents, "agent_id", "5 doublons techniques")
]:
    if col in df.columns:
        s = df[col]
        n = int((s.notna() & s.duplicated()).sum())
        synthese.append({
            "controle": f"copies excédentaires {nom}",
            "nombre": n,
            "taux_pct": taux_pct(n, len(df)),
            "attendu": attendu
        })

synthese_eda = pd.DataFrame(synthese)
display(synthese_eda)

## Graphiques de synthèse

In [ ]:
# 6) Visualiser les anomalies quantitatives
if not synthese_eda.empty:
    graphique = synthese_eda.dropna(subset=["nombre"]).copy()
    plt.figure(figsize=(11, max(5, 0.45 * len(graphique))))
    plt.barh(graphique["controle"], graphique["nombre"])
    plt.xlabel("Nombre de lignes / cas détectés")
    plt.title(
        "Synthèse des anomalies détectées "
        "(les dénominateurs diffèrent selon le contrôle)"
    )
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

# Valeurs manquantes par fichier
taux_manquants = pd.DataFrame({
    nom: (df.isna().mean() * 100)
    for nom, df in datasets.items()
}).fillna(0)

plt.figure(figsize=(12, 5))
sns.heatmap(taux_manquants.T, cmap="YlOrRd", annot=False)
plt.title("Pourcentage de valeurs manquantes par colonne et par fichier")
plt.xlabel("Colonnes (positions regroupées par fichier dans les lignes)")
plt.ylabel("Fichier")
plt.tight_layout()
plt.show()

## Conclusion EDA — modèle à compléter après exécution

À la fin de l'exécution du notebook, rédige une conclusion à partir du tableau de synthèse et des graphiques :

- **Doublons :** préciser le nombre de copies excédentaires et le nombre de lignes concernées pour chaque identifiant.
- **Montants :** indiquer les montants négatifs, nuls et manquants. Le dépassement d'un plafond individuel ne pourra être validé qu'après obtention de cette règle métier ; les plafonds quotidiens/mensuels ne sont pas interchangeables avec un plafond par transaction.
- **MSISDN :** utiliser le taux de numéros invalides après normalisation syntaxique, en traitant séparément les numéros absents lorsqu'ils sont autorisés.
- **Intégrité référentielle :** documenter les transactions pointant vers un agent inexistant, sans considérer automatiquement l'absence d'agent comme une anomalie pour les opérations sans agent.
- **Antifraude :** documenter les paires miroir candidates et la fenêtre de vélocité. Ces détections sont des alertes à investiguer, pas des preuves de fraude.
- **Référentiels clients/agents :** distinguer les groupes de doublons des copies supplémentaires, puis décider de la stratégie de dédoublonnage à l'étape de nettoyage.

Ne supprime pas les lignes à ce stade : conserve les données brutes et prépare une étape distincte de nettoyage/validation pour le pipeline ETL.